# Attention & PAA Attribution Gallery

Interpretability visualizations for the trained StrokeGAT model.

**PAA (Eq. 12):** A_ij = alpha_ij * (p_i^T . p_j)
- alpha_ij: GAT attention coefficient
- p_i, p_j: connectivity probability vectors encoding neighborhood lesion-type composition

Visualizations:
1. Attention heatmaps by layer
2. Attention by atlas region
3. PAA attribution maps
4. Stroke core vs penumbra detection
5. PAA score distributions
6. 3D interactive PAA graph

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from stroke_gat.config import load_config
from stroke_gat.data.service import DataService
from stroke_gat.models.gat import StrokeGAT
from stroke_gat.training.callbacks import ModelCheckpoint
from stroke_gat.visualization.attention_maps import (
    project_attention_to_volume,
    plot_attention_heatmap,
    plot_attention_by_layer,
    plot_attention_by_region,
)
from stroke_gat.visualization.attribution import (
    plot_paa_attribution_map,
    plot_stroke_penumbra_detection,
    plot_graph_attention_attribution,
    plot_paa_weight_distribution,
)

In [ ]:
# Load config and trained model
config = load_config("../configs/default.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load a pre-computed graph file
graph_dir = Path(config.paths.output_graphs)
graph_files = sorted(graph_dir.glob("*_supervoxel_graph.pt"))
print(f"Found {len(graph_files)} graph files")

data = torch.load(str(graph_files[0]), weights_only=False)
print(f"Loaded graph: {data.num_nodes} nodes, {data.num_edges} edges")

# Load trained model
checkpoint_path = Path("../outputs/checkpoints/best_model.pt")
in_dim = data.x.size(1)
model = StrokeGAT(in_channels=in_dim, config=config.model)
ModelCheckpoint.load(checkpoint_path, model)
model.to(device)
model.eval()

print(f"Model loaded from {checkpoint_path}")

In [ ]:
# Run inference with attention extraction
data_dev = data.to(device)
with torch.no_grad():
    logits, attention_weights, paa_scores = model.get_attention_maps(
        data_dev.x,
        data_dev.edge_index,
        connectivity_probs=getattr(data_dev, "connectivity_probs", None),
    )

preds = logits.argmax(dim=1).cpu()
print(f"Predictions: {torch.bincount(preds, minlength=3).tolist()}")
print(f"Ground truth: {torch.bincount(data.y, minlength=3).tolist()}")
print(f"Attention layers extracted: {len(attention_weights)}")
for i, attn in enumerate(attention_weights):
    print(f"  Layer {i}: shape={attn.shape}, range=[{attn.min():.4f}, {attn.max():.4f}]")
print(f"PAA scores: {'available' if paa_scores is not None else 'N/A'}")

In [ ]:
# Load the subject's volume data for overlay visualizations
service = DataService(config.paths)
subject_id = graph_files[0].stem.replace("_supervoxel_graph", "")
print(f"Subject: {subject_id}")

modalities_nifti = service.load_subject_modalities(subject_id)
t1_data = modalities_nifti["T1"].get_fdata(dtype=np.float32) if "T1" in modalities_nifti else None
atlas_data, _ = service.load_atlas()
atlas_labels = service.load_atlas_labels()
masks = service.load_subject_masks(subject_id)

# Supervoxel labels may be stored in the graph Data object
supervoxel_labels = getattr(data, "supervoxel_labels", None)
if supervoxel_labels is not None:
    supervoxel_labels = supervoxel_labels.numpy()
    print(f"Supervoxel labels: {supervoxel_labels.shape}")
else:
    print("Note: supervoxel_labels not in graph. Volume projections will be skipped.")

In [ ]:
# Attention heatmaps by layer (projected to volume)
if supervoxel_labels is not None and t1_data is not None:
    fig = plot_attention_by_layer(
        [aw.cpu() for aw in attention_weights],
        data.edge_index,
        supervoxel_labels,
        t1_data,
    )
    plt.show()
else:
    print("Skipping volume attention heatmaps (no supervoxel_labels)")

In [ ]:
# Top attended atlas regions
if supervoxel_labels is not None and t1_data is not None:
    attn_volume = project_attention_to_volume(
        attention_weights[-1].cpu(), data.edge_index, supervoxel_labels
    )
    fig = plot_attention_by_region(attn_volume, atlas_data, atlas_labels, top_n=15)
    plt.show()
else:
    print("Skipping region attention analysis")

In [ ]:
# PAA attribution map overlaid on brain
if paa_scores is not None and supervoxel_labels is not None and t1_data is not None:
    fig = plot_paa_attribution_map(
        paa_scores.cpu(),
        data.edge_index,
        supervoxel_labels,
        t1_data,
        title=f"PAA Attribution Map - {subject_id}",
    )
    plt.show()
else:
    print("Skipping PAA map")

In [ ]:
# Stroke core vs penumbra detection
if paa_scores is not None and supervoxel_labels is not None and t1_data is not None:
    lesion_mask = masks.get("General", np.zeros_like(t1_data, dtype=np.uint8))

    fig = plot_stroke_penumbra_detection(
        paa_scores.cpu(),
        data.edge_index,
        supervoxel_labels,
        lesion_mask,
        t1_data,
        core_threshold=0.7,
        penumbra_threshold=0.3,
    )
    plt.show()
else:
    print("Skipping stroke detection visualization")

In [ ]:
# PAA score distribution by edge type
if paa_scores is not None:
    fig = plot_paa_weight_distribution(
        paa_scores.cpu(),
        data.edge_index,
        data.y,
    )
    plt.show()

In [ ]:
# 3D interactive graph colored by PAA attribution
if paa_scores is not None:
    fig_3d = plot_graph_attention_attribution(
        data,
        paa_scores.cpu(),
        title=f"PAA Attribution Graph - {subject_id}",
        node_size=4.0,
    )
    fig_3d.show()

## Clinical Interpretation

### Attention Patterns
- **Layer 1 (early):** Broad attention on local intensity differences
- **Layer 2 (middle):** Selective focus on normal/abnormal tissue boundaries
- **Layer 3 (late):** Concentrated on lesion-relevant regions

### PAA Attribution
- **High PAA edges:** Most informative message-passing pathways for stroke detection
- **Stroke core (red):** High PAA within lesion mask = confident detection
- **Penumbra (yellow):** Moderate PAA around lesion boundaries = tissue at risk

### Edge Type Analysis
- Lesion-lesion edges should have higher PAA than normal-normal
- Mixed (lesion-normal) edges capture boundary information

The PAA module (Eq. 12) modulates attention by A_ij = alpha_ij * (p_i^T . p_j),
where p_i are connectivity probability vectors encoding each node's neighborhood
lesion-type composition.